# Chat with PDF: a simple RAG pipeline (Deloitte JD)

Ask questions about a PDF and get answers **grounded in the document, with page citations**.

**RAG (Retrieval-Augmented Generation)** in 4 steps:
1. **Ingest:** read the PDF page by page and split the text into small overlapping chunks
2. **Embed:** turn every chunk into a vector with **FastEmbed** (`BAAI/bge-small-en-v1.5`, runs locally on CPU)
3. **Retrieve:** embed the question and find the most similar chunks with **cosine similarity** over an **in-memory NumPy index**
4. **Generate:** send only those chunks to an LLM (**Groq**, free tier) with a prompt that says *answer only from the context and cite pages*

Test document: the *Deloitte US-India Consultative Offerings, Analyst or Equivalent* job description (3 pages).

## 1. Setup

In [21]:
import os
import re
import textwrap
from pathlib import Path

import numpy as np
from pypdf import PdfReader
from fastembed import TextEmbedding
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a .env file if there is one

EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # small, fast, 384-dim embeddings
LLM_MODEL = "openai/gpt-oss-120b"        # served free on Groq
CHUNK_WORDS = 120                        # words per chunk
OVERLAP_WORDS = 30                       # words shared between neighbouring chunks
TOP_K = 4                                # chunks sent to the LLM

## 2. Load the PDF (page by page, so we can cite pages)

In [22]:
# Use the Deloitte JD from the data/ folder
PDF_NAME = "Deloitte_US-India_Consultative_Offerings_JD.pdf"
data_dir = next(p for p in [Path("data"), Path("../data")] if p.exists())
PDF_PATH = data_dir / PDF_NAME

def load_pdf(path):
    pages = []
    for page_no, page in enumerate(PdfReader(path).pages, start=1):
        text = page.extract_text() or ""
        text = text.replace("Copyright © 2026 Deloitte Development LLC. All rights reserved.", "")  # footer
        text = text.replace("ﬁ", "fi")              # fix the 'fi' ligature (e.g. "Benefits")
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            pages.append({"page": page_no, "text": text})
    return pages

pages = load_pdf(PDF_PATH)
print(f"{PDF_PATH.name}: {len(pages)} pages with text")
print("\nPage 1 preview:", pages[0]["text"][:300], "...")

Deloitte_US-India_Consultative_Offerings_JD.pdf: 3 pages with text

Page 1 preview: Job Title: Analyst or Equivalent – Job Description Are you a quick learner with a willingness to work with new technologies with an analytical mindset? At Deloitte, we help provide insights to clients who are one of the pioneers in the fields of technology. Here, innovative practices are aimed to be ...


## 3. Chunking
Each page is split into chunks of ~120 words with a 30-word overlap, so a sentence cut at a chunk boundary still appears whole in one of the two chunks. Each chunk keeps its page number.

In [23]:
def chunk_pages(pages, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    chunks = []
    step = size - overlap
    for p in pages:
        words = p["text"].split()
        for start in range(0, len(words), step):
            piece = words[start:start + size]
            if len(piece) < 15 and start > 0:   # skip tiny tail pieces
                break
            chunks.append({"page": p["page"], "text": " ".join(piece)})
    return chunks

chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks created")
for c in chunks[:3]:
    print(f"[page {c['page']}] {c['text'][:120]}...")

13 chunks created
[page 1] Job Title: Analyst or Equivalent – Job Description Are you a quick learner with a willingness to work with new technolog...
[page 1] information and technology. We assist in accomplishing this mission through our capabilities, continued development of o...
[page 1] and productivity. • AI/ML and Generative AI insights: Help identify business opportunities and implementation requiremen...


## 4. Embeddings + in-memory vector index
FastEmbed turns text into vectors. Vectors are normalised to length 1, so the **dot product equals cosine similarity**. The whole index is just a NumPy matrix: one row per chunk.

The first run downloads the embedding model (~130 MB), which takes a minute (already cached if you ran the Cognizant notebook).

In [24]:
embedder = TextEmbedding(model_name=EMBED_MODEL)

def normalise(m):
    return m / np.linalg.norm(m, axis=1, keepdims=True)

chunk_texts = [c["text"] for c in chunks]
index = normalise(np.array(list(embedder.passage_embed(chunk_texts))))
print("Index shape:", index.shape, "(chunks x embedding dimensions)")

Index shape: (13, 384) (chunks x embedding dimensions)


## 5. Retrieval

In [25]:
def search(question, k=TOP_K):
    q = normalise(np.array(list(embedder.query_embed([question]))))[0]
    scores = index @ q                      # cosine similarity with every chunk
    best = np.argsort(scores)[::-1][:k]     # top-k highest scores
    return [{**chunks[i], "score": float(scores[i])} for i in best]

for hit in search("What kind of AI work will I do in this role?"):
    print(f"score={hit['score']:.3f}  page {hit['page']}: {hit['text'][:100]}...")

score=0.750  page 2: • Enterprise platforms and client enablement: Support in implementing and configuring packaged enter...
score=0.719  page 1: and productivity. • AI/ML and Generative AI insights: Help identify business opportunities and imple...
score=0.700  page 1: information and technology. We assist in accomplishing this mission through our capabilities, contin...
score=0.694  page 1: Help develop proofs of concept, experiment with emerging AI/ML technologies, support research and de...


## 6. Generation with an LLM (Groq)
The prompt tells the model to answer **only** from the retrieved context, cite pages like `[p. 2]`, and say so when the answer is not in the document. This reduces hallucination.

**API key:** get a free key at https://console.groq.com/keys and put it in a `.env` file in the project folder:
```
GROQ_API_KEY=your_key_here
```
Without a key, the notebook still runs and shows the retrieved passages instead of an LLM answer.

In [26]:
PROMPT = """You are a helpful assistant answering questions about a document.
Use ONLY the context below. Cite the page for each fact like [p. 2].
If the answer is not in the context, reply exactly: "I could not find that in the document."

Context:
{context}

Question: {question}
Answer:"""

api_key = os.getenv("GROQ_API_KEY")
client = None
if api_key:
    from groq import Groq
    client = Groq(api_key=api_key)
    print("Groq client ready:", LLM_MODEL)
else:
    print("No GROQ_API_KEY found: answers will show retrieved passages only.")

def ask(question, k=TOP_K, show_sources=True):
    hits = search(question, k)
    context = "\n\n".join(f"[p. {h['page']}] {h['text']}" for h in hits)
    print("Q:", question)
    if client:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            temperature=0,
            messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
        )
        answer = response.choices[0].message.content.strip()
        print("A:", textwrap.fill(answer, 110))
    else:
        print("A: (no LLM key) top passage ->", textwrap.shorten(hits[0]["text"], 300))
    if show_sources:
        print("Sources:", ", ".join(f"p. {h['page']} ({h['score']:.2f})" for h in hits))
    print("-" * 110)

Groq client ready: openai/gpt-oss-120b


## 7. Try it

In [27]:
ask("What is the job title and what kind of work will I do?")
ask("Which AI/ML and Generative AI capabilities are mentioned?")
ask("What are the key job responsibilities?")
ask("What general expectations and skills are they looking for?")
ask("Will I have to work night shifts or weekends?")

Q: What is the job title and what kind of work will I do?
A: **Job Title:** Analyst or Equivalent [p. 1]  **Kind of work you’ll do:** You will provide advisory and
implementation support to clients, helping design and develop insights across industries using Digital, Cloud,
ERP packages, Analytics, AI/ML, Generative AI and agentic AI. Your responsibilities include implementing and
configuring enterprise platforms (planning, procurement, manufacturing, product development, quality),
interpreting AI/ML model outputs, analyzing and improving business and support processes to meet strategic
client objectives, and delivering client presentations as part of independent or team‑based work [p. 2].
Sources: p. 2 (0.64), p. 1 (0.63), p. 2 (0.63), p. 2 (0.60)
--------------------------------------------------------------------------------------------------------------
Q: Which AI/ML and Generative AI capabilities are mentioned?
A: The document lists the following AI/ML and Generative AI capabilit

### Guardrail check: questions the document cannot answer
A good RAG system should refuse instead of making something up. The JD does **not** state a salary, so the model should say it could not find it.

In [28]:
ask("What is the salary for this role?")
ask("Who won the cricket world cup in 2011?")

Q: What is the salary for this role?
A: I could not find that in the document.
Sources: p. 2 (0.64), p. 2 (0.61), p. 1 (0.57), p. 3 (0.52)
--------------------------------------------------------------------------------------------------------------
Q: Who won the cricket world cup in 2011?
A: I could not find that in the document.
Sources: p. 2 (0.40), p. 3 (0.38), p. 3 (0.37), p. 2 (0.36)
--------------------------------------------------------------------------------------------------------------


## 8. Evaluate retrieval
A tiny test set: each question with the page that contains the answer. **Hit@k** = share of questions where the correct page is in the top-k retrieved chunks. **MRR** (mean reciprocal rank) rewards finding it at rank 1.

In [29]:
test_set = [
    ("Which graph technologies such as knowledge graphs and graph neural networks are used?", 1),
    ("What does the data, analytics and engineering work involve?", 1),
    ("What research and innovation work, like proofs of concept, is expected?", 1),
    ("Which enterprise platform processes like procurement and manufacturing are supported?", 2),
    ("Is working in night shifts or on weekends required?", 2),
    ("What are the three consulting competencies?", 2),
    ("Where is DU The Leadership Center in India located?", 3),
    ("What is Deloitte Touche Tohmatsu Limited (DTTL)?", 3),
]

hits_at_1, hits_at_k, reciprocal_ranks = 0, 0, []
for question, page in test_set:
    pages_found = [h["page"] for h in search(question, TOP_K)]
    rank = pages_found.index(page) + 1 if page in pages_found else None
    hits_at_1 += rank == 1
    hits_at_k += rank is not None
    reciprocal_ranks.append(1 / rank if rank else 0)
    print(f"{'OK ' if rank else 'MISS'} rank={rank}  expected p.{page}  got {pages_found}  | {question}")

n = len(test_set)
print(f"\nHit@1 = {hits_at_1 / n:.0%}   Hit@{TOP_K} = {hits_at_k / n:.0%}   MRR = {np.mean(reciprocal_ranks):.2f}")

OK  rank=1  expected p.1  got [1, 1, 2, 1]  | Which graph technologies such as knowledge graphs and graph neural networks are used?
OK  rank=1  expected p.1  got [1, 1, 2, 2]  | What does the data, analytics and engineering work involve?
OK  rank=1  expected p.1  got [1, 1, 1, 2]  | What research and innovation work, like proofs of concept, is expected?
OK  rank=1  expected p.2  got [2, 1, 1, 2]  | Which enterprise platform processes like procurement and manufacturing are supported?
OK  rank=1  expected p.2  got [2, 2, 2, 3]  | Is working in night shifts or on weekends required?
OK  rank=1  expected p.2  got [2, 2, 2, 1]  | What are the three consulting competencies?
OK  rank=1  expected p.3  got [3, 3, 1, 2]  | Where is DU The Leadership Center in India located?
OK  rank=1  expected p.3  got [3, 3, 3, 3]  | What is Deloitte Touche Tohmatsu Limited (DTTL)?

Hit@1 = 100%   Hit@4 = 100%   MRR = 1.00


## 9. Ask your own question

In [30]:
ask("What professional development and benefits does Deloitte offer?")

Q: What professional development and benefits does Deloitte offer?
A: Deloitte provides a range of professional‑development resources, including diverse learning and networking
opportunities that are designed to accelerate careers and build leadership skills.  The firm also highlights
its state‑of‑the‑art DU: The Leadership Center in Hyderabad, India, as a tangible symbol of its commitment to
the holistic growth and development of its people [p. 3].  In terms of benefits, Deloitte offers a
comprehensive rewards program that is intended to help professionals thrive mentally, physically and
financially while living their purpose.  A broad suite of benefits is available to employees and their loved
ones, with eligibility based on factors such as role, tenure, and type of employment [p. 3].
Sources: p. 3 (0.79), p. 3 (0.77), p. 2 (0.72), p. 3 (0.71)
--------------------------------------------------------------------------------------------------------------


## Summary
| Step | Choice | Why |
|---|---|---|
| Loading | pypdf, page by page | keeps page numbers for citations |
| Chunking | 120 words, 30 overlap | small enough to be specific, overlap keeps sentences whole |
| Embeddings | FastEmbed `bge-small-en-v1.5` | free, local, fast on CPU |
| Vector index | NumPy matrix, cosine similarity | the document is small, so no vector database is needed |
| LLM | Groq `gpt-oss-120b`, temperature 0 | free, fast, deterministic answers |
| Guardrail | "answer only from context" + refusal message | reduces hallucination (e.g. salary is not in this JD) |
| Evaluation | Hit@k and MRR on a small labelled set | checks that retrieval finds the right page |

**Possible improvements:** hybrid search (BM25 + vectors), a reranker, a vector database (Qdrant/FAISS) for many documents, and a Streamlit UI.